# 🎨 Fooocus Designer 2.0 — Turnkey Graphic Design Appliance
### Professional Commercial Graphic Asset Production on Google Colab (Free T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiL4gh/Fooocus-Design-Tool/blob/main/Fooocus_Designer_2_0.ipynb)

---

### ⚡ What is Fooocus Designer 2.0?
A zero-friction professional graphic design workstation powered by **SDXL**, **Juggernaut-XL-v9**, and **SDXL-Lightning**.

- 🎯 **Modern WebApp Workstation:** Clean, full-width glassmorphism UI built for commercial workflows.
- 📦 **Safe Batch Generation:** Generate up to 32 images per batch with progressive real-time canvas streaming.
- 🖼️ **Pro Gallery & Quick Actions:** 1-click **Remove Background** (transparent PNG) and **Vectorize SVG** (instant download).
- 🛡️ **Stealth AI Metadata Cleanser:** Automatically strips AI prompts and injects standard Adobe 300 DPI metadata for microstock approval.
- 🪄 **AI Prompt Copilot:** One-click prompt expander that optimizes composition, lighting, vector standards, and negative prompts.
- ⚡ **Dual-Speed Engine:** **⚡ Fast (~3s)** for instant iteration vs **🎯 Master (~15s)** for full LoRA precision.
- 🏷️ **Baked Category LoRAs:** Silhouette, Flat Vector, Sticker, Logo, Pattern, and Poster with zero reload penalties.
- 🤖 **Curated Base Models:** Juggernaut XL v9, RealVisXL v4, Animagine XL 3.1, DreamShaper XL Turbo, SDXL Base 1.0, SDXL Turbo.
- 📦 **12 Mockup Products & 6 Lighting Styles:** Test your designs on apparel, mugs, packaging, billboards, and devices.

> **⚠️ Important GPU Requirement:** Ensure GPU acceleration is enabled:
> **Runtime** ➔ **Change runtime type** ➔ **Hardware accelerator** ➔ **T4 GPU**.

In [ ]:
#@title 1. Setup Environment & Dependencies (Run First)
#@markdown Verifies GPU and installs dependencies cleanly before runtime initialization.
import subprocess, os

# 1. Verify GPU
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode('utf-8').strip()
    print(f"✅ GPU Detected: {gpu_info}")
except Exception:
    print("⚠️ Warning: No GPU detected! Please ensure you selected Runtime -> Change runtime type -> T4 GPU.")

# 2. Prepare repo directory
%cd /content
if os.path.exists('/content/Fooocus-Design-Tool/Fooocus-Design-Tool'):
    !rm -rf /content/Fooocus-Design-Tool/Fooocus-Design-Tool

if not os.path.exists('/content/Fooocus-Design-Tool/launch.py'):
    print("\n📦 Cloning Fooocus-Design-Tool...")
    !git clone https://github.com/NiL4gh/Fooocus-Design-Tool.git
else:
    print("\n📦 Fetching latest updates from GitHub...")
    %cd /content/Fooocus-Design-Tool
    !git reset --hard
    !git pull origin main
    %cd /content

%cd /content/Fooocus-Design-Tool

# 3. Clean up conflicting packages and install requirements
!pip uninstall -y torchao -q
print("\n📥 Installing dependencies (fast caching)...")
!pip install -r requirements.txt --quiet
print("\n✅ Environment and dependencies are ready! Now run Cell 2 below.")

In [ ]:
#@title 2. Launch Fooocus Designer 2.0 (1-Click Launch)
#@markdown Select your preferred setup options:
Preload_Models = False #@param {type:"boolean"}
Use_Google_Drive_For_Outputs = True #@param {type:"boolean"}

import os, sys
from IPython.display import Audio, HTML, display

# 1. Background Tab Anti-Timeout Guard (keeps Colab active while you work in webapp)
display(HTML('<div style="background: rgba(46, 213, 115, 0.12); border-left: 4px solid #2ed573; padding: 10px 14px; border-radius: 6px; margin-bottom: 12px; font-weight: 600; color: #2ed573; font-family: sans-serif;">🛡️ Anti-Timeout Guard Active: Background audio keepalive is running. Google Colab will not sleep while you use the WebApp tab.</div>'))
display(Audio('data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEARKwAAIhYAQACABAAZGF0YQAAAAA=', autoplay=True))

# 2. Ensure repo exists and is up to date (even if Cell 1 was skipped)
%cd /content
if not os.path.exists('/content/Fooocus-Design-Tool/launch.py'):
    print("📦 Cloning Fooocus-Design-Tool...")
    !git clone https://github.com/NiL4gh/Fooocus-Design-Tool.git
    %cd /content/Fooocus-Design-Tool
    !pip uninstall -y torchao -q
    !pip install -r requirements.txt --quiet
else:
    %cd /content/Fooocus-Design-Tool
    !git reset --hard
    !git pull origin main

%cd /content/Fooocus-Design-Tool

# 3. Setup zero-disconnect Cloudflare Tunnel binary
if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/usr/bin/cloudflared'):
    print("🌐 Setting up rock-solid Cloudflare tunnel (zero disconnects)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 4. Mount Google Drive if requested (placed AFTER pip to prevent numpy import warnings)
output_dir_symlink = None
if Use_Google_Drive_For_Outputs:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("📁 Mounting Google Drive to preserve generated assets...")
            drive.mount('/content/drive')
        else:
            print("📁 Google Drive is already mounted.")
        drive_out = '/content/drive/MyDrive/Fooocus_Designer_Outputs'
        os.makedirs(drive_out, exist_ok=True)
        output_dir_symlink = drive_out
        print(f"✅ Generated assets will be saved directly to: {drive_out}")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}. Saving to local session instead.")

# 5. Link outputs to Google Drive if mounted
if output_dir_symlink:
    if os.path.exists('outputs') and not os.path.islink('outputs'):
        !rm -rf outputs
    if not os.path.exists('outputs'):
        try:
            os.symlink(output_dir_symlink, 'outputs')
            print("🔗 Outputs symlinked to Google Drive.")
        except Exception:
            pass

# 6. Optional Preload (only if user explicitly checks the box)
if Preload_Models:
    print("\n" + "=" * 60)
    print("📦 Pre-downloading Base Model: RunDiffusion/Juggernaut-XL-v9 (SDXL FP16 ~6.6GB)")
    print("⚡ Pre-downloading Speed Adapter: ByteDance/SDXL-Lightning (4-step LoRA ~390MB)")
    print("=" * 60)
    !python -u -c "from modules.sdxl_pipeline import load_pipeline, unload_pipeline; load_pipeline(speed_mode='fast'); unload_pipeline(); print('\n✅ Models cached!')"

# 7. Launch web UI (prints both Cloudflare Tunnel and Gradio Public links)
print("\n🚀 Launching Fooocus Designer 2.0...")
print("=" * 60)
!python -u launch.py --share

In [ ]:
#@title 3. (Optional) Setup ngrok Tunnel for Rock-Solid Persistent URL
#@markdown If the standard Gradio live link disconnects during long sessions, use an ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok_token = "" #@param {type:"string"}

if ngrok_token.strip():
    %cd /content/Fooocus-Design-Tool
    !pip install pyngrok --quiet
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token.strip())
    tunnel = ngrok.connect(7865)
    print(f"\n🌐 Public ngrok URL: {tunnel.public_url}")
    print("🚀 Launching Fooocus Designer 2.0...")
    !python -u launch.py --no-share
else:
    print("ℹ️ Paste your ngrok auth token above if you wish to use an ngrok tunnel.")